# 

In [12]:
from ogb.utils.features import (allowable_features, atom_to_feature_vector,
 bond_to_feature_vector, atom_feature_vector_to_dict, bond_feature_vector_to_dict) 
import numpy as np
from tqdm import tqdm
import instructions_smol
import datasets
from datasets import load_dataset
import pandas as pd
import os
from rdkit import Chem
from ood_dataset_download import get_data_list

In [13]:
alchemy_path = '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_alchemy1k_0405'
aqsol_path = '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_aqsol_0405'
orderly_forward_path = '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_orderly-forward_reaction_prediction_0405'
presto_forward_path = '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_presto-forward_reaction_prediction_0405'
orderly_retro_path = '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_orderly-retrosynthesis_0405'
presto_retro_path = '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_presto-retrosynthesis_0405'

In [18]:
test_path = '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_gap_fixed_0224'

In [19]:
test_data = datasets.load_from_disk(test_path)
test_data

Dataset({
    features: ['task', 'x', 'edge_index', 'edge_attr', 'additional_x', 'additional_edge_index', 'additional_edge_attr', 'prompt_text', 'target_text', 'input_mol_string'],
    num_rows: 32828
})

In [20]:
set(test_data['task'])

{'bace',
 'chebi-20-mol2text',
 'chebi-20-text2mol',
 'forward_reaction_prediction',
 'qm9_homo',
 'qm9_homo_lumo_gap',
 'qm9_lumo',
 'reagent_prediction',
 'retrosynthesis',
 'smol-forward_synthesis',
 'smol-molecule_captioning',
 'smol-molecule_generation',
 'smol-property_prediction-bbbp',
 'smol-property_prediction-clintox',
 'smol-property_prediction-esol',
 'smol-property_prediction-hiv',
 'smol-property_prediction-lipo',
 'smol-property_prediction-sider',
 'smol-retrosynthesis'}

In [ ]:
prompts = [i['prompt_text'] for i in test_data]
exceptions = []
for i in range(len(prompts)):
    if '[/INST]' not in prompts[i]:
        exceptions.append(i)
print(exceptions)

[]


In [11]:
prompts = [i['prompt_text'] for i in test_ood_data]
exceptions = []
for i in range(len(prompts)):
    if '[/INST]' not in prompts[i]:
        exceptions.append(i)
print(exceptions)

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221,

In [9]:
concated_test_data = datasets.concatenate_datasets([test_ood_data, test_data])
concated_test_data

Dataset({
    features: ['task', 'x', 'edge_index', 'edge_attr', 'additional_x', 'additional_edge_index', 'additional_edge_attr', 'input_mol_string', 'prompt_text', 'target_text'],
    num_rows: 36757
})

In [10]:
concated_test_data.save_to_disk('/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_gap_fixed_0330')

Saving the dataset (1/1 shards): 100%|██████████| 36757/36757 [00:04<00:00, 7657.92 examples/s]


In [ ]:
# load data
alchemy_data = datasets.load_from_disk(alchemy_path)
aqsol_data = datasets.load_from_disk(aqsol_path)
orderly_forward_data = datasets.load_from_disk(orderly_forward_path)
presto_forward_data = datasets.load_from_disk(presto_forward_path)
orderly_retro_data = datasets.load_from_disk(orderly_retro_path)
presto_retro_data = datasets.load_from_disk(presto_retro_path)


FileNotFoundError: Directory /data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_orderly-forward_reaction_prediction_0405 not found

In [6]:
ood_data = datasets.concatenate_datasets(
    [
        alchemy_homo_data,
        aqsol_data,
        orderly_forward_data,
        presto_forward_data,
    ]
)

In [7]:
ood_data

Dataset({
    features: ['task', 'x', 'edge_index', 'edge_attr', 'additional_x', 'additional_edge_index', 'additional_edge_attr', 'input_mol_string', 'prompt_text', 'target_text'],
    num_rows: 3929
})

In [9]:
ood_data.save_to_disk(
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_ood_0328'
)

Saving the dataset (1/1 shards): 100%|██████████| 3929/3929 [00:06<00:00, 593.09 examples/s] 


In [8]:
test_data_path = '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_augmented_0211'
test_data = datasets.load_from_disk(test_data_path)

In [9]:
test_data

Dataset({
    features: ['task', 'x', 'edge_index', 'edge_attr', 'additional_x', 'additional_edge_index', 'additional_edge_attr', 'prompt_text', 'target_text', 'input_mol_string', '0-th_rejected_x', '0-th_rejected_edge_index', '0-th_rejected_edge_attr', '0-th_additional_rejected_x', '0-th_additional_rejected_edge_index', '0-th_additional_rejected_edge_attr', '1-th_rejected_x', '1-th_rejected_edge_index', '1-th_rejected_edge_attr', '1-th_additional_rejected_x', '1-th_additional_rejected_edge_index', '1-th_additional_rejected_edge_attr', '2-th_rejected_x', '2-th_rejected_edge_index', '2-th_rejected_edge_attr', '2-th_additional_rejected_x', '2-th_additional_rejected_edge_index', '2-th_additional_rejected_edge_attr', '3-th_rejected_x', '3-th_rejected_edge_index', '3-th_rejected_edge_attr', '3-th_additional_rejected_x', '3-th_additional_rejected_edge_index', '3-th_additional_rejected_edge_attr', '4-th_rejected_x', '4-th_rejected_edge_index', '4-th_rejected_edge_attr', '4-th_additional_rejec

In [10]:
ood_paths = [
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_alchemy_homo',
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_aqsol',
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_orderly-forward_reaction_prediction',
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_presto-forward_reaction_prediction',
]
test_paths = [
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_qm9_homo_0219',
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_smol-property_prediction-esol_0219',
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_smol-property_prediction-lipo_0219',
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_bace_0219',
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_smol-property_prediction-bbbp_0219',
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_smol-property_prediction-clintox_0219',
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_smol-property_prediction-hiv_0219',
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_smol-property_prediction-sider_0219',
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_forward_reaction_prediction_0219',
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_chebi-20-mol2text_0219',
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_chebi-20-text2mol_0219',
]

train_paths = [
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_train_qm9_homo_0219',
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_train_smol-property_prediction-esol_0219',
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_train_smol-property_prediction-lipo_0219',
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_train_bace_0219',
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_train_smol-property_prediction-bbbp_0219',
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_train_smol-property_prediction-clintox_0219',
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_train_smol-property_prediction-hiv_0219',
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_train_smol-property_prediction-sider_0219',
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_train_forward_reaction_prediction_0219',
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_train_chebi-20-mol2text_0219',
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_train_chebi-20-text2mol_0219',
]


In [11]:
test_data = datasets.concatenate_datasets(
    [datasets.load_from_disk(path) for path in test_paths + ood_paths]
)

In [12]:
test_data

Dataset({
    features: ['task', 'x', 'edge_index', 'edge_attr', 'additional_x', 'additional_edge_index', 'additional_edge_attr', 'input_mol_string', 'prompt_text', 'target_text'],
    num_rows: 20205
})

In [17]:
test_data.save_to_disk(
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_ablation_0224'
)
test_data.save_to_disk(
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_validation_ablation_0224'
)

Saving the dataset (0/1 shards):   0%|          | 0/20205 [00:00<?, ? examples/s]

Saving the dataset (1/1 shards): 100%|██████████| 20205/20205 [00:01<00:00, 11982.98 examples/s]


In [14]:
train_data = datasets.concatenate_datasets(
    [datasets.load_from_disk(path) for path in train_paths]
)

In [15]:
train_data

Dataset({
    features: ['task', 'x', 'edge_index', 'edge_attr', 'additional_x', 'additional_edge_index', 'additional_edge_attr', 'input_mol_string', 'prompt_text', 'target_text'],
    num_rows: 361115
})

In [18]:
train_data.save_to_disk(
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_train_ablation_0224'
)

Saving the dataset (0/7 shards):   0%|          | 0/361115 [00:00<?, ? examples/s]

Saving the dataset (7/7 shards): 100%|██████████| 361115/361115 [00:28<00:00, 12500.32 examples/s]
